NAME : VIKAS MALVIYA

SCH. NO. : 25215011109

SUBJECT : LAB 04

TITLE: WISDM_DADS



In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
def sliding_window(data, sensor_cols, label_col, window_size=128, overlap=0.5):
    step = int(window_size * (1 - overlap))
    X, y = [], []

    for i in range(0, len(data) - window_size, step):
        window = data.iloc[i:i+window_size]
        X.append(window[sensor_cols].values)
        y.append(window[label_col].mode()[0])

    return np.array(X), np.array(y)

In [ ]:
class HARDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class HARTransformer(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=64, nhead=4, layers=2):
        super().__init__()

        self.embedding = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [ ]:
def train_model(model, loader, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            out = model(X)
            preds = torch.argmax(out, dim=1)

            y_true.extend(y.numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    return acc, f1

#WISDM

In [ ]:
wisdm = pd.read_csv(
    "/content/time_series_data_human_activities.csv",
    header=None,
    names=["user", "activity", "timestamp", "ax", "ay", "az"],
    comment=';'
)


/tmp/ipython-input-3156593475.py:1: DtypeWarning: Columns (0,2,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  wisdm = pd.read_csv(


In [ ]:
wisdm["ax"] = pd.to_numeric(wisdm["ax"], errors="coerce")
wisdm["ay"] = pd.to_numeric(wisdm["ay"], errors="coerce")
wisdm["az"] = pd.to_numeric(wisdm["az"], errors="coerce")

wisdm = wisdm.dropna()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_w = StandardScaler()
wisdm[["ax", "ay", "az"]] = scaler_w.fit_transform(
    wisdm[["ax", "ay", "az"]]
)

In [ ]:
le_w = LabelEncoder()
wisdm["activity"] = le_w.fit_transform(wisdm["activity"])

scaler_w = StandardScaler()
wisdm[["ax", "ay", "az"]] = scaler_w.fit_transform(wisdm[["ax", "ay", "az"]])

In [ ]:
X_w, y_w = sliding_window(
    wisdm,
    sensor_cols=["ax", "ay", "az"],
    label_col="activity"
)

In [ ]:
ds_w = HARDataset(X_w, y_w)
dl_w = DataLoader(ds_w, batch_size=32, shuffle=True)

model_w = HARTransformer(
    input_dim=3,
    num_classes=len(np.unique(y_w))
).to(device)

train_model(model_w, dl_w)
wisdm_acc, wisdm_f1 = evaluate_model(model_w, dl_w)

print("WISDM Accuracy:", wisdm_acc)
print("WISDM F1:", wisdm_f1)

Epoch 1/10 | Loss: 277.7443
Epoch 2/10 | Loss: 159.9449
Epoch 3/10 | Loss: 118.4818
Epoch 4/10 | Loss: 99.2038
Epoch 5/10 | Loss: 86.4957
Epoch 6/10 | Loss: 76.4978
Epoch 7/10 | Loss: 69.7030
Epoch 8/10 | Loss: 65.5808
Epoch 9/10 | Loss: 57.0169
Epoch 10/10 | Loss: 57.8791
WISDM Accuracy: 0.9694765708835102
WISDM F1: 0.9695839243311966


#DADS

In [ ]:
import os
import numpy as np
import pandas as pd

DADS_PATH = "/content/drive/MyDrive/MANIT LAB/Gen AI/DADS dataset"

dads_data = []

for activity in sorted(os.listdir(DADS_PATH)):
    activity_path = os.path.join(DADS_PATH, activity)
    if not os.path.isdir(activity_path):
        continue

    for person in os.listdir(activity_path):
        person_path = os.path.join(activity_path, person)
        if not os.path.isdir(person_path):
            continue

        for file in os.listdir(person_path):
            if not file.endswith(".txt"):
                continue

            file_path = os.path.join(person_path, file)

            # Load sensor data (acc + gyro)
            signal = np.loadtxt(
                file_path,
                delimiter=',',
                usecols=range(6)
            )

            df = pd.DataFrame(
                signal,
                columns=['ax','ay','az','gx','gy','gz']
            )

            df['activity'] = activity
            dads_data.append(df)

dads_df = pd.concat(dads_data, ignore_index=True)

print(dads_df.head())
print("DADS shape:", dads_df.shape)


       ax      ay      az        gx        gy        gz activity
0  7.9213  1.2261  5.7118 -0.004102  0.025824 -0.007636      a01
1  7.8392  1.2413  5.7710  0.010322  0.014988 -0.005750      a01
2  7.8917  1.2267  5.6146  0.030156  0.002412  0.005271      a01
3  7.8617  1.1750  5.6215 -0.010519  0.018843  0.006965      a01
4  7.8769  1.3004  5.7343  0.017574  0.008538 -0.015741      a01
DADS shape: (1140000, 7)


In [ ]:
print(dads_df.dtypes)


ax          float64
ay          float64
az          float64
gx          float64
gy          float64
gz          float64
activity     object
dtype: object


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

le_d = LabelEncoder()
dads_df["activity"] = le_d.fit_transform(dads_df["activity"])

scaler_d = StandardScaler()
dads_df[["ax","ay","az","gx","gy","gz"]] = scaler_d.fit_transform(
    dads_df[["ax","ay","az","gx","gy","gz"]]
)


In [ ]:
print(wisdm.dtypes)
print(dads_df.dtypes)

print(wisdm.head())
print(dads_df.head())


user          object
activity       int64
timestamp     object
ax           float64
ay           float64
az           float64
dtype: object
ax          float64
ay          float64
az          float64
gx          float64
gy          float64
gz          float64
activity      int64
dtype: object
  user  activity      timestamp        ax        ay        az
1    1         5  4991922345000  0.002594  0.514975 -0.507894
2    1         5  4991972333000  0.893168  0.015937 -0.188432
3    1         5  4992022351000  0.037292 -0.252891 -0.188432
4    1         5  4992072339000 -0.402212 -0.344975 -0.228104
5    1         5  4992122358000 -0.760755 -0.451912 -0.491190
         ax        ay        az        gx        gy        gz  activity
0  0.027587  0.776636  0.831752 -0.001644  0.017552 -0.013915         0
1  0.013025  0.782431  0.848484  0.016522  0.001871 -0.007846         0
2  0.022337  0.776865  0.804281  0.041501 -0.016328  0.027618         0
3  0.017016  0.757155  0.806231 -0.009726  0.0

In [ ]:
X_d, y_d = sliding_window(
    dads_df,
    sensor_cols=["ax","ay","az","gx","gy","gz"],
    label_col="activity"
)


In [ ]:
ds_d = HARDataset(X_d, y_d)
dl_d = DataLoader(ds_d, batch_size=32, shuffle=True)

model_d = HARTransformer(
    input_dim=6,
    num_classes=len(np.unique(y_d))
).to(device)

train_model(model_d, dl_d)
dads_acc, dads_f1 = evaluate_model(model_d, dl_d)

print("DADS Accuracy:", dads_acc)
print("DADS F1:", dads_f1)


Epoch 1/10 | Loss: 355.7562
Epoch 2/10 | Loss: 113.6108
Epoch 3/10 | Loss: 81.0514
Epoch 4/10 | Loss: 69.6107
Epoch 5/10 | Loss: 55.9466
Epoch 6/10 | Loss: 48.4956
Epoch 7/10 | Loss: 54.7907
Epoch 8/10 | Loss: 38.9392
Epoch 9/10 | Loss: 45.0127
Epoch 10/10 | Loss: 39.7534
DADS Accuracy: 0.982538880467127
DADS F1: 0.9825326698355199


In [ ]:
results = pd.DataFrame({
    "Dataset": ["WISDM", "DADS"],
    "Sensors": ["Accelerometer", "Accelerometer + Gyroscope"],
    "Accuracy": [wisdm_acc, dads_acc],
    "F1-Score": [wisdm_f1, dads_f1]
})

results


,Dataset,Sensors,Accuracy,F1-Score
0,WISDM,Accelerometer,0.969477,0.969584
1,DADS,Accelerometer + Gyroscope,0.982539,0.982533
